In [1]:

import cv2
import os
import numpy as np

face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')


def face_embedding_dataset(root_dir: str):
    X = []
    Y = []

    files = os.listdir(root_dir)
    for file in files:
        file_path = os.path.join(root_dir, file)

        for img_name in os.listdir(file_path):
            img_path = os.path.join(file_path, img_name)

            img = cv2.imread(img_path)
            img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # RGB -> GRAY
            # detecting face with .xml, 1.3 : scaling down img 30% at each step
            faces = face_cascade.detectMultiScale(img_gray, 1.3, 5)
            # 5 is minimum neighbours, meaning how many each candiate rectangle should considered as faces

            for x, y, w, h in faces:
                face = img_gray[y:y+h, x:x+w]
                face = cv2.resize(face, (100, 100))  # resizing the face

                X.append(face.flatten())  # flatten into 2500D vector (1x2500)
                Y.append(file)

    return np.array(X), np.array(Y)


root_dir = 'face_data_3'
X, Y = face_embedding_dataset(root_dir=root_dir)

In [2]:
print(X)

[[155 155 156 ... 178 167 149]
 [166 166 166 ... 239 240 239]
 [169 169 169 ...  65  55  24]
 ...
 [ 62  64  87 ...  41  50  62]
 [ 85  88 110 ...  29  22  32]
 [ 71  77  89 ...  24  26  35]]


In [3]:
print(Y)

['Abhinandan' 'Abhinandan' 'Abhinandan' ... 'Yazhini' 'Yazhini' 'Yazhini']


In [4]:
from sklearn.neighbors import KNeighborsClassifier
import pickle  # for saving up the file

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X, Y)

with open('LF_model_3.pkl', 'wb') as f:
    pickle.dump(knn, f)

In [ ]:
# import cv2
# import pickle
# import numpy as np

# with open('LF_model_3.pkl', 'rb') as f:
#     knn = pickle.load(f)

# face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')
# cam = cv2.VideoCapture(0)

# while True:
#     ret, frame = cam.read()
#     cam_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#     cam_faces = face_cascade.detectMultiScale(cam_gray, 1.3, 5)

#     for x, y, w, h in cam_faces:
#         face = cam_gray[y:y+h, x:x+w]
#         face = cv2.resize(face, (100, 100)).flatten().reshape(1, -1)
#         face_pred = knn.predict(face)[0]

#         cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
#         cv2.putText(frame, face_pred, (x, y - 10),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

#     cv2.imshow("Face Recognition", frame)

#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break

# cam.release()
# cv2.destroyAllWindows()
